<a href="https://colab.research.google.com/github/TheideaLAXE/IntelligentSystems/blob/main/NoteGenie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [140]:
# ============================================================================
# 1. INSTALLATION & SETUP
# ============================================================================
!pip install -q -U google-generativeai gradio pypdf streamlit

import gradio as gr
import google.generativeai as genai
from google.colab import userdata
import pypdf
import logging # Import logging
import os # Import os for os.makedirs
from logging.handlers import TimedRotatingFileHandler # Import TimedRotatingFileHandler
import streamlit as st # Import streamlit

# Configure basic logging to a file with rotation
log_dir = './logs'
os.makedirs(log_dir, exist_ok=True)

# Clear any existing handlers to ensure our new setup is effective
logging.getLogger().handlers = []

# Create a logger
logger = logging.getLogger()
logger.setLevel(logging.DEBUG) # Set to DEBUG for verbose logging

# Create a rotating file handler
log_file_path = os.path.join(log_dir, 'note_genie.log')
handler = TimedRotatingFileHandler(log_file_path, when='midnight', interval=1, backupCount=7)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
logger.addHandler(handler)

logging.info("✅ NoteGenie: Logging configured with daily rotation.") # Add a test log message here

# Attempt to load your secret key
try:
    API_KEY = userdata.get('GenieKey')
    genai.configure(api_key=API_KEY)
    logging.info("✅ NoteGenie: Gemini API Key successfully loaded!") # Changed to logging
except Exception as e:
    logging.error(f"❌ ERROR: 'GenieKey' not found in Colab Secrets or other API Key error: {e}") # Changed to logging, added exception info

logging.debug(f"Log file path for rotation: {os.path.abspath(log_file_path)}")

In [141]:


# ============================================================================
# 2. CONFIGURATION
# ============================================================================
class NoteGenieConfig:
    MODELS = {"fast": "gemini-1.5-flash", "quality": "gemini-1.5-pro"}
    STUDY_PROMPT = """You are NoteGenie. Convert the following text into clear,
    hierarchical bullet points. Use **Bold** for key terms.

    TEXT: {text}"""

In [142]:
# ============================================================================
# 3. CORE LOGIC
# ============================================================================
import os
import logging # Import logging if not already present from cell 1

class NoteGenie:
    def __init__(self):
        self.config = NoteGenieConfig()
        self.upload_dir = "./uploads"
        os.makedirs(self.upload_dir, exist_ok=True)
        logging.info(f"Upload directory created at: {self.upload_dir}")

    def list_uploaded_files(self):
        """Lists files in the uploads directory."""
        files = [f for f in os.listdir(self.upload_dir) if os.path.isfile(os.path.join(self.upload_dir, f))]
        return files

    def generate(self, text_input, selected_file_name, model_choice):
        text_to_process = ""

        # Prioritize selected file from uploads folder
        if selected_file_name and selected_file_name != "None (direct text input)": # Updated condition for Streamlit dropdown
            file_path = os.path.join(self.upload_dir, selected_file_name)
            if not os.path.exists(file_path):
                logging.error(f"Selected file not found: {file_path}")
                return f"❌ Error: Selected file '{selected_file_name}' not found in uploads folder."
            try:
                if selected_file_name.lower().endswith('.pdf'):
                    reader = pypdf.PdfReader(file_path)
                    text_to_process = "".join([page.extract_text() for page in reader.pages])
                    logging.info(f"Extracted text from PDF file: {selected_file_name}")
                else:
                    with open(file_path, "r", encoding="utf-8") as f:
                        text_to_process = f.read()
                    logging.info(f"Read text from file: {selected_file_name}")
            except Exception as e:
                logging.error(f"Error reading file {selected_file_name}: {str(e)}")
                return f"❌ Error reading file '{selected_file_name}': {str(e)}"
        elif text_input and text_input.strip(): # Use direct text input if no file selected or "None" selected
            text_to_process = text_input
            logging.info("Processing direct text input.")

        if not text_to_process or not text_to_process.strip():
            logging.warning("No text entered or file selected.")
            return "⚠️ Please enter some text or select a file from uploads folder!"

        try:
            # Removed gradio.Progress() dependency
            model = genai.GenerativeModel(model_name=self.config.MODELS[model_choice])
            logging.info(f"Generating content with model: {model_choice}")
            response = model.generate_content(self.config.STUDY_PROMPT.format(text=text_to_process))
            logging.info("Content generation successful.")
            return response.text.strip()
        except Exception as e:
            logging.error(f"Error during content generation: {str(e)}")
            return f"❌ Error: {str(e)}"

In [143]:
# ============================================================================
# 4. STREAMLIT INTERFACE
# ============================================================================
def create_streamlit_ui():
    st.set_page_config(page_title="NoteGenie", layout="centered")

    app = NoteGenie()

    st.title("📚 NoteGenie")
    st.markdown("### Private AI Study Assistant")
    st.markdown("**Instructions:** Paste your study material in the text box below, or select a file from the 'uploads' folder. Select your preferred model, then click 'Generate' to get a hierarchical study guide.")

    # Input for text
    input_text = st.text_area("Paste Notes Here", height=150)

    # Dropdown for uploaded files
    uploaded_files = app.list_uploaded_files()
    uploaded_files_options = ["None (direct text input)"] + uploaded_files # Add option for direct text
    selected_file_name = st.selectbox("Or Select File from Uploads Folder", uploaded_files_options)

    # Model selection
    model_choice = st.selectbox("Select Model", list(app.config.MODELS.keys()), index=0)

    # Generate button
    if st.button("✨ Generate"): # Assign a key to the button if there are multiple buttons on the page.
        if not input_text.strip() and selected_file_name == "None (direct text input)":
            st.warning("⚠️ Please enter some text or select a file from the uploads folder!")
        else:
            with st.spinner("Consulting Gemini..."):
                output = app.generate(input_text, selected_file_name, model_choice)
                st.text_area("Study Guide", value=output, height=400)

# This will be called when the Streamlit app is run
if __name__ == "__main__":
    create_streamlit_ui()

# Save this cell's content to a file to be run by Streamlit
with open('streamlit_app.py', 'w') as f:
    f.write('''
import streamlit as st
import google.generativeai as genai
from google.colab import userdata
import pypdf
import logging
import os
from logging.handlers import TimedRotatingFileHandler

# Re-import necessary components from other cells if they are not in `streamlit_app.py`
# Assuming NoteGenieConfig and NoteGenie class definitions are available (e.g., copied or imported from another file)
# For simplicity, copy the relevant class definitions here for standalone execution

# Configuration Class (from uwZhSTnAcnSB)
class NoteGenieConfig:
    MODELS = {"fast": "gemini-1.5-flash", "quality": "gemini-1.5-pro"}
    STUDY_PROMPT = """You are NoteGenie. Convert the following text into clear,
    hierarchical bullet points. Use **Bold** for key terms.

    TEXT: {text}"""

# NoteGenie Class (from rMDDQl9ycupH)
class NoteGenie:
    def __init__(self):
        self.config = NoteGenieConfig()
        self.upload_dir = "./uploads"
        os.makedirs(self.upload_dir, exist_ok=True)
        # Logging setup is typically done once globally, so we won't duplicate it here
        # logging.info(f"Upload directory created at: {self.upload_dir}") # Avoid re-logging during app execution

    def list_uploaded_files(self):
        """Lists files in the uploads directory."""
        files = [f for f in os.listdir(self.upload_dir) if os.path.isfile(os.path.join(self.upload_dir, f))]
        return files

    def generate(self, text_input, selected_file_name, model_choice):
        text_to_process = ""

        if selected_file_name and selected_file_name != "None (direct text input)":
            file_path = os.path.join(self.upload_dir, selected_file_name)
            if not os.path.exists(file_path):
                # Use st.error for Streamlit UI feedback
                st.error(f"❌ Error: Selected file '{selected_file_name}' not found in uploads folder.")
                return f"❌ Error: Selected file '{selected_file_name}' not found in uploads folder."
            try:
                if selected_file_name.lower().endswith('.pdf'):
                    reader = pypdf.PdfReader(file_path)
                    text_to_process = "".join([page.extract_text() for page in reader.pages])
                else:
                    with open(file_path, "r", encoding="utf-8") as f:
                        text_to_process = f.read()
            except Exception as e:
                st.error(f"❌ Error reading file '{selected_file_name}': {str(e)}")
                return f"❌ Error reading file '{selected_file_name}': {str(e)}"
        elif text_input and text_input.strip():
            text_to_process = text_input

        if not text_to_process or not text_to_process.strip():
            st.warning("⚠️ Please enter some text or select a file from uploads folder!")
            return "⚠️ Please enter some text or select a file from uploads folder!"

        try:
            # API_KEY should be configured globally or passed
            # For a standalone Streamlit script, you might need to load it here again
            # genai.configure(api_key=userdata.get('GenieKey')) # Re-configure if not already done
            model = genai.GenerativeModel(model_name=self.config.MODELS[model_choice])
            response = model.generate_content(self.config.STUDY_PROMPT.format(text=text_to_process))
            return response.text.strip()
        except Exception as e:
            st.error(f"❌ Error during content generation: {str(e)}")
            return f"❌ Error: {str(e)}"

# Main Streamlit UI function
def create_streamlit_ui():
    st.set_page_config(page_title="NoteGenie", layout="centered")

    app = NoteGenie()

    st.title("📚 NoteGenie")
    st.markdown("### Private AI Study Assistant")
    st.markdown("**Instructions:** Paste your study material in the text box below, or select a file from the 'uploads' folder. Select your preferred model, then click 'Generate' to get a hierarchical study guide.")

    input_text = st.text_area("Paste Notes Here", height=150)

    uploaded_files = app.list_uploaded_files()
    uploaded_files_options = ["None (direct text input)"] + uploaded_files
    selected_file_name = st.selectbox("Or Select File from Uploads Folder", uploaded_files_options)

    model_choice = st.selectbox("Select Model", list(app.config.MODELS.keys()), index=0)

    if st.button("✨ Generate"):
        if not input_text.strip() and selected_file_name == "None (direct text input)":
            st.warning("⚠️ Please enter some text or select a file from the uploads folder!")
        else:
            with st.spinner("Consulting Gemini..."):
                output = app.generate(input_text, selected_file_name, model_choice)
                st.text_area("Study Guide", value=output, height=400)

if __name__ == "__main__":
    # API Key setup should be in the main Colab notebook, and genai configured once.
    # However, if running this .py file directly, you'd need this here.
    try:
        API_KEY = userdata.get('GenieKey')
        genai.configure(api_key=API_KEY)
    except Exception as e:
        st.error(f"❌ ERROR: 'GenieKey' not found in Colab Secrets or other API Key error: {e}")
        st.stop() # Stop Streamlit app if API key is not found

    create_streamlit_ui()
'''
)


2026-04-01 11:05:16.060 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:05:16.073 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:05:16.076 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:05:16.079 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:05:16.084 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:05:16.087 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:05:16.091 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 11:05:16.095 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [144]:
# ============================================================================
# 5. RUN STREAMLIT APP
# ============================================================================
!streamlit run streamlit_app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.145.8.45:8501

  Stopping...
  Stopping...


In [145]:
import os

log_dir = './logs'
log_file_path = os.path.join(log_dir, 'note_genie.log')

if os.path.exists(log_file_path):
    with open(log_file_path, 'r') as f:
        log_content = f.read()
    print(log_content)
else:
    print(f"Log file not found at: {log_file_path}")

2026-04-01 10:23:14,990 - INFO - ✅ NoteGenie: Logging configured.
2026-04-01 10:23:22,851 - INFO - ✅ NoteGenie: Gemini API Key successfully loaded!
2026-04-01 10:23:22,851 - DEBUG - Log file path: /content/logs/note_genie.log
2026-04-01 10:23:43,599 - INFO - ✅ NoteGenie: Logging configured.
2026-04-01 10:23:44,425 - INFO - ✅ NoteGenie: Gemini API Key successfully loaded!
2026-04-01 10:23:44,426 - DEBUG - Log file path: /content/logs/note_genie.log
2026-04-01 10:23:44,522 - INFO - Upload directory created at: ./uploads
2026-04-01 10:23:44,529 - DEBUG - close.started
2026-04-01 10:23:44,534 - DEBUG - close.complete
2026-04-01 10:23:44,537 - DEBUG - connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=None socket_options=None
2026-04-01 10:23:44,603 - DEBUG - connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7e9a417ab0b0>
2026-04-01 10:23:44,608 - DEBUG - start_tls.started ssl_context=<ssl.SSLContext object at 0x7e9a420cbbd0> server

In [146]:

# ============================================================================
# 2. CONFIGURATION
# ============================================================================
class NoteGenieConfig:
    MODELS = {"fast": "gemini-1.5-flash", "quality": "gemini-1.5-pro"}
    STUDY_PROMPT = """You are NoteGenie. Convert the following text into clear,
    hierarchical bullet points. Use **Bold** for key terms.

    TEXT: {text}"""